# QR 모듈 격자 피싱 분류 — Colab 실행 노트북이 노트북에는 로직을 쓰지 않는다. 전부 `qrphish` 패키지 함수 호출이고 노트북은 설정·실행·표시만 한다.셀 5 이후는 중단·재개가 가능하다 (`results.json`이 이미 있는 조합은 자동으로 건너뛴다).

산출물(`artifacts/`, `reports/`)은 학습 중 Google Drive에 직접 기록되므로 Colab 세션이 끊겨도 보존된다.

## [1] 설치

In [ ]:
!pip -q install "git+https://github.com/bnbong/CNN-QR-phishing-detector.git@main"

## [2] GPU 확인

In [ ]:
import torch

print(torch.__version__, torch.cuda.is_available())

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## [3] 데이터 경로 확정Drive 마운트 또는 Kaggle 다운로드 중 하나를 쓰고, 어느 쪽이든 `DATA_CSV` 변수 하나로 수렴시킨다.

In [ ]:
import os

# (A) Google Drive

from google.colab import drive

drive.mount('/content/drive')

DATA_CSV = "/content/drive/MyDrive/qrphish/webphish.csv"

OUT_DIR = "/content/drive/MyDrive/qrphish_out"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# (B) Kaggle — kaggle.json 업로드 후
# from google.colab import files; files.upload()
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d guchiopara/look-before-you-leap -p /content/data --unzip# DATA_CSV = "/content/data/<파일명>.csv"
# OUT_DIR = "/content/drive/MyDrive/qrphish_out"  # Drive 마운트 시와 동일하게 정의 필요

## [4] config 로드

In [ ]:
!git clone -q https://github.com/bnbong/CNN-QR-phishing-detector.git /content/repo
%cd /content/repo

from qrphish.config import load_config
from qrphish.runner import apply_overrides

# Config는 frozen dataclass다. 필드를 직접 대입하지 말고 점 표기 오버라이드를 쓴다.
cfg = apply_overrides(
    load_config("configs/base.yaml"),
    {
        "data.csv_path": DATA_CSV,
        "output_dir": OUT_DIR + "/artifacts",
        "reports_dir": OUT_DIR + "/reports",
    },
)
cfg

## [5] P0 — 모델 학습 전에 반드시 통과시킨다층 카운트 표, `first_pad_codeword` 클래스별 분포, 그룹 분할 진단, 라운드트립 게이트.**여기서 어떤 층이 primary인지 눈으로 확인하고 진행 여부를 결정한다.**

In [ ]:
from qrphish.runner import run_p0

p0 = run_p0(cfg)


print("roundtrip gate:", p0["roundtrip_gate"]["passed"], p0["roundtrip_gate"]["n_checked"])

import pandas as pd

pd.DataFrame(p0["stratum_counts"]["cells"])

In [ ]:
from IPython.display import Image

Image(p0["first_pad_png"])

## [6] P1 — 주 실험

In [ ]:
from qrphish.runner import run_matrix

_ = run_matrix(cfg, "P1")

## [7] P2 — 절제

In [ ]:
_ = run_matrix(cfg, "P2")

## [8] P3 — 해석 (Grad-CAM)

In [ ]:
from qrphish.runner import run_explain, condition_id

best_condition = "norm-exact-data_only-off-small_cnn"   # mask-off 최고 성능 조건
explain_result = run_explain(cfg, best_condition)
explain_result

## [9] 결과 저장 + 표 미리보기

In [ ]:
from qrphish.runner import aggregate_reports

tables = aggregate_reports(cfg)
tables

In [ ]:
import pandas as pd

pd.read_csv(tables["main"])

산출물은 학습 중 Drive에 직접 기록되므로 세션이 끊겨도 보존되며, 재실행 시 results.json이 있는 조건은 건너뛴다.

In [ ]:
!ls -la $OUT_DIR/reports
!du -sh $OUT_DIR/artifacts